# Step 1 - PCAPNG to flow CSV (one CSV per capture)

**What this notebook does**

Takes the raw MSCAD captures in `C:\MSCAD2022\PCAPNG_ORIGINAL` and produces one
flow CSV per capture in `C:\MSCAD2022\CSVs`, with the **6 flow identifiers the
Kaggle `MSCAD.csv` does not have** (Src IP, Dst IP, Src Port, Dst Port, Protocol,
Timestamp) plus the same 66 CICFlowMeter features and a Label - 73 columns.

**It never touches the originals.** `PCAPNG_ORIGINAL\` and `CSV_ORIGINAL\` are
read-only. Everything written goes to `PCAPS\`, `CSVs\` or `Notebook\outputs\step1\`.

**Where the labels come from.** From the *documented* MSCAD capture -> attack
mapping (`config.CAPTURE_LABELS`) plus the attacker IPs *measured* from the
traffic (any Src IP sending at least `ATTACKER_THRESHOLD` x the top sender's
forward packets). Flows in an attack capture that do not involve an attacker IP
are labelled `Benign`. **Nothing is ever inferred from packet contents.**

---

## Install everything first (cmd, NOT PowerShell)

```
"C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m pip install scapy numpy scipy pandas requests tqdm jupyter pytest
"C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m pip install cicflowmeter==0.5.0 --no-deps --ignore-requires-python
```

`cicflowmeter 0.5.0` pins old dependencies and declares `python_requires < 3.13`,
so `--no-deps` and `--ignore-requires-python` are both required. Wireshark must
also be installed - it ships `editcap.exe`, `tshark.exe` and `capinfos.exe`.

## Start Jupyter and run the tests (cmd)

```
cd /d C:\MSCAD2022\Notebook
"C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m jupyter notebook
"C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m pytest tests -v
```

## Folder layout

```
C:\MSCAD2022\
  PCAPNG_ORIGINAL\   input .pcapng        READ-ONLY
  PCAPS\             Stage B output       .pcap files
  CSVs\              Stage C/D output     <stem>_flows.csv   (+ raw\ before renaming)
  CSV_ORIGINAL\      Kaggle MSCAD.csv     READ-ONLY, reference only
  Notebook\          all code (this file, config.py, src\, tests\, outputs\step1\)
```

Every cell is safe to re-run. All the logic is in `src\converter.py`; the cells
below only call it.

## 1. Setup

Point Python at `C:\MSCAD2022\Notebook`, turn on autoreload (so edits to
`src\converter.py` take effect without restarting the kernel), and import.

In [ ]:
import sys

# Jupyter's working folder is already C:\MSCAD2022\Notebook, but adding it
# explicitly means the imports work even if you started Jupyter elsewhere.
sys.path.insert(0, r"C:\MSCAD2022\Notebook")

%load_ext autoreload
%autoreload 2

import config
from src import converter

print("python     :", sys.version.split()[0])
print("config     :", config.__file__)
print("converter  :", converter.__file__)

## 2. Config - every path, and whether it exists

`config.py` is the only file with hard-coded paths. If a row says `NO` for a
folder that should exist, fix it in `config.py` before going further.

In [ ]:
rep = converter.Report()
converter.stage_a_paths(rep)

print()
print("flow timeout       :", config.FLOW_TIMEOUT_S, "s")
print("idle timeout       :", config.IDLE_TIMEOUT_S, "s")
print("overwrite outputs  :", config.OVERWRITE)
print("progress every     :", f"{config.PROGRESS_EVERY:,}", "packets")
print("attacker threshold :", config.ATTACKER_THRESHOLD, "x the top sender")
print("output columns     :", len(config.FLOW_COLUMNS),
      f"({len(config.ID_COLUMNS)} identifiers + {len(config.KAGGLE_FEATURES)} features + Label)")

## 3. Tool check

Checks Python 3.13, the packages, `jupyter`, `pytest`, and `editcap` / `tshark` /
`capinfos` (`-v`). If something is missing it prints the exact cmd command to fix
it. `capinfos` is optional - it only makes packet counting faster.

In [ ]:
rep = converter.Report()
missing = converter.stage_a_environment(rep)

print()
print("missing:", missing if missing else "nothing - environment is ready")

## 4. Scan and inventory

Finds every `.pcapng` / `.pcap` under `PCAPNG_ORIGINAL` (recursive,
case-insensitive, so `.PCAPNG` matches too) and shows size, the label each stem
will get, and whether any two captures share a stem.

In [ ]:
rep = converter.Report()

capture_dir, why = converter.locate_capture_dir(rep)
print("capture folder:", capture_dir)
print("reason        :", why)
print()

captures = converter.find_captures(capture_dir) if capture_dir else []
converter.stage_a_inventory(rep, captures)

## 5. Kaggle reference and column diff

Reads `CSV_ORIGINAL\MSCAD.csv` (read-only) and prints its shape, its numbered
column names exactly as stored, and its Label counts. Then diffs those names
against `config.KAGGLE_FEATURES`.

**The Kaggle header always wins.** If anything differs, this cell prints a
ready-to-paste `KAGGLE_FEATURES = [...]` block for `config.py`.

In [ ]:
rep = converter.Report()

kaggle_df, kaggle_cols = converter.read_kaggle(rep)
rep.header("column names: config vs the Kaggle header")
kaggle_features = converter.compare_headers(rep, kaggle_cols)

## 6. PCAPNG to PCAP, with a packet-count check

`editcap -F pcap` per capture. A file that is already `.pcap` is **copied**, never
moved. Outputs that already exist are skipped unless `config.OVERWRITE = True`.

Each output is then verified: **the source and the converted file must have the
same packet count.** A mismatch is reported as `failed`, not as a warning - a
format conversion that loses packets is a broken conversion.

The table is saved to `outputs\step1\pcap_inventory.csv`.

In [ ]:
import pandas as pd

rep = converter.Report()
rows = converter.convert_all(captures, rep=rep)          # uses config.OVERWRITE

stage_b = pd.DataFrame(rows)
inventory_path = converter.write_inventory(captures)
print()
print("inventory saved to:", inventory_path)
stage_b

## 7. N-0 test run

> **Stage C - not implemented yet.** The flow extractor (`converter.extract_flows`)
> is the next thing to be written. Everything it needs is already in place and
> unit-tested: the slim `PacketRecord`, the `flow_key` that keeps ICMP flows
> (ports 0, protocol 1), the TCP-flag helpers, the 120 s / 240 s timeouts, and the
> whole rename / reorder / microsecond / labelling chain.
>
> This cell is safe to run now - it reports the status instead of raising.

In [ ]:
try:
    n0 = next(c for c in captures if c.stem == "N-0")
except StopIteration:
    print("N-0 not found in the inventory - check cell 4 first.")
else:
    pcap_path = config.PCAP_DIR / converter.output_pcap_name(n0, captures)
    print("would extract flows from:", pcap_path)
    try:
        raw = converter.extract_flows(pcap_path)
        df, info = converter.postprocess(raw, "N-0")
        assert list(df.columns) == config.FLOW_COLUMNS, "header does not match FLOW_COLUMNS"
        print("flows:", len(df), "| shape:", df.shape)
        display(df.head())
        print(df[config.LABEL_COLUMN].value_counts())
    except NotImplementedError as exc:
        print()
        print("STAGE C PENDING:", exc)

## 8. Full conversion - all six captures

> **Stage C/D - not implemented yet** (same reason as cell 7). Once the extractor
> exists, this cell runs all six captures with a tqdm bar and a progress line every
> `config.PROGRESS_EVERY` packets, and writes `outputs\step1\conversion_log.csv`
> and `outputs\step1\label_report.csv`.

In [ ]:
try:
    results = [converter.postprocess(converter.extract_flows(
                   config.PCAP_DIR / converter.output_pcap_name(c, captures)), c.stem)
               for c in captures if c.stem in config.CAPTURE_LABELS]
    print("converted", len(results), "captures")
except NotImplementedError as exc:
    print("STAGE C/D PENDING:", exc)

## 9. Column validation

Asserts that **every** CSV in `C:\MSCAD2022\CSVs` has exactly
`config.FLOW_COLUMNS`, in exactly that order. Run it after cell 8. With no CSVs
yet it simply says so.

In [ ]:
import pandas as pd

csv_files = sorted(config.CSV_OUTPUT_DIR.glob("*_flows.csv"))
if not csv_files:
    print("no *_flows.csv in", config.CSV_OUTPUT_DIR, "- run cell 8 first (needs Stage C).")
else:
    all_ok = True
    for p in csv_files:
        header = list(pd.read_csv(p, nrows=0).columns)
        ok = header == config.FLOW_COLUMNS
        all_ok &= ok
        print(f"{p.name:<28} {len(header):>3} columns  {'OK' if ok else 'MISMATCH'}")
        if not ok:
            print("   missing:", [c for c in config.FLOW_COLUMNS if c not in header])
            print("   extra  :", [c for c in header if c not in config.FLOW_COLUMNS])
    assert all_ok, "at least one CSV does not match config.FLOW_COLUMNS"
    print()
    print(f"all {len(csv_files)} CSVs match config.FLOW_COLUMNS exactly")

## 10. Verification table

> **Stage D - not implemented yet.** It will check, per capture: packets read;
> IPv4 packets in the flows (`Tot Fwd Pkts + Tot Bwd Pkts`) against an independent
> `tshark -Y ip` count (these must match **exactly**, the gap to the total being
> ARP/IPv6); flows written twice (must be 0); NaN / negative / min>max / invalid
> ports (must be 0); `inf` in `Flow Byts/s` and `Flow Pkts/s` for zero-duration
> flows (counted, **not** fixed); rows with identical 66-feature values (counted,
> **not** dropped). Saved to `outputs\step1\verification.csv`.

In [ ]:
verification_path = config.VERIFICATION_CSV
if verification_path.exists():
    import pandas as pd
    display(pd.read_csv(verification_path))
else:
    print("not written yet:", verification_path)
    print("Needs Stage D, which needs the Stage C extractor.")

## 11. Summary and logs

Lists what exists in `outputs\step1\`, `PCAPS\` and `CSVs\` right now.

In [ ]:
def summarise(folder, pattern="*"):
    files = sorted(p for p in folder.glob(pattern) if p.is_file())
    print(f"{folder}  ({len(files)} file(s))")
    for p in files:
        print(f"    {p.name:<34} {p.stat().st_size / 1024:>10.1f} KB")
    if not files:
        print("    (empty)")
    print()

summarise(config.OUTPUT_DIR)
summarise(config.PCAP_DIR, "*.pcap")
summarise(config.CSV_OUTPUT_DIR, "*_flows.csv")
summarise(config.RAW_DIR, "*_raw.csv")

## 12. Preview one CSV

Head, shape, dtypes and Label counts for the first output CSV.

In [ ]:
import pandas as pd

csv_files = sorted(config.CSV_OUTPUT_DIR.glob("*_flows.csv"))
if not csv_files:
    print("no *_flows.csv yet in", config.CSV_OUTPUT_DIR)
else:
    p = csv_files[0]
    df = pd.read_csv(p, low_memory=False)
    print(p.name, "| shape:", df.shape)
    print()
    print("dtypes:")
    print(df.dtypes.value_counts())
    print()
    print("Label counts:")
    print(df[config.LABEL_COLUMN].value_counts())
    display(df.head())